In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap
import pandas as pd
import numpy as np
from scipy.spatial.distance import squareform, pdist

from joblib import Parallel, delayed
import sys
import json
sys.path.append("../src/irl") 
import data_corruption_utils

In [ ]:
with open("../../../dir_paths.json") as f:
    config = json.load(f)
    dataframes_path = config["dataframes_path"]
    project_output_path = config["project_output_path"]
    
sampled_matched_perturbed_df = pd.read_pickle(dataframes_path + "/sampled_matched_perturbed_df_final.pkl")


In [ ]:
# Example distance function between two matrices
def euclidean_dist(A, B):
    # flatten then L2 distance
    return np.linalg.norm(A - B)


def compute_row(i, X):
    n = len(X)
    row = np.zeros(n)
    for j in range(i+1, n):
        row[j] = euclidean_dist(X[i], X[j])
    return i, row
    
def weighted_symmetric_kl_divergence(p, q, p_counts, q_counts):
    p_sum = np.sum(p_counts, axis=1) 
    q_sum = np.sum(q_counts, axis=1)

    pw = p_sum / np.sum(p_sum)
    qw = q_sum / np.sum(q_sum)


    # Ensure p and q are numpy arrays
    p = np.asarray(p)
    q = np.asarray(q)

    eps = 1e-10
    p = np.clip(p, eps, 1)
    q = np.clip(q, eps, 1)
    
    # Calculate KL divergence
    p_kl = p * np.log(p / q)
    q_kl = q * np.log(q / p)
    
    kl_div_pq = pw * np.sum(p_kl,axis=1)     
    kl_div_qp = qw * np.sum(q_kl,axis=1)

    # Calculate symmetric KL divergence
    symmetric_kl = (np.sum(kl_div_pq) + np.sum(kl_div_qp)) / 2.0
    
    return symmetric_kl


In [ ]:
df = sampled_matched_perturbed_df[sampled_matched_perturbed_df.perturb_percent == 0.0]
mask = df["troll"] == 1
df_a1 = df[mask].drop_duplicates(subset="user", keep="first")
df_other = df[~mask]
df_no_dup = pd.concat([df_a1, df_other], ignore_index=True)

In [ ]:
df_no_dup["emp_pol"] =  df_no_dup['traj_counts_perturbed'].apply(data_corruption_utils.normalize_replace_zeros)

In [ ]:
X = df_no_dup.emp_pol.values
labels = df_no_dup.russian.values

In [ ]:
n = len(X)
dist_matrix = np.zeros((n, n))

# Parallelize rows
results = Parallel(n_jobs=-1, backend="loky")(
    delayed(compute_row)(i, X) for i in range(n)
)

# Fill results back into matrix
for i, row in results:
    dist_matrix[i, i+1:] = row[i+1:]
    dist_matrix[i+1:, i] = row[i+1:]

In [ ]:

# Run t-SNE with precomputed distances
tsne = TSNE(n_components=2, metric="precomputed", init="random", random_state=42)
X_embedded = tsne.fit_transform(dist_matrix)


In [ ]:
colors = np.array(['blue', 'red'])  # map 0 -> blue, 1 -> red
point_colors = colors[labels]

plt.figure(figsize=(8,6))
plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=point_colors,s=40, alpha=0.2)
# Create a legend
for label, color in zip(['organic', 'troll'],['blue', 'red']):
    plt.scatter([], [], c=color, label=f'{label}')  # empty scatter for legend
plt.legend(title="Labels")
plt.title("t-SNE with Euclidean Distance")
plt.savefig(f"tsne_euclidean.pdf", bbox_inches='tight')
plt.show()

In [ ]:


# Run UMAP with precomputed distances
umap_model = umap.UMAP(
    n_components=2,
    metric="precomputed",
    random_state=42
)
X_embedded = umap_model.fit_transform(dist_matrix)
colors = np.array(['blue', 'red'])  # map 0 -> blue, 1 -> red
point_colors = colors[labels]

plt.figure(figsize=(8,6))
plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=point_colors,s=40, alpha=0.2)
# Create a legend
for label, color in zip(['organic', 'troll'],['blue', 'red']):
    plt.scatter([], [], c=color, label=f'{label}')  # empty scatter for legend
plt.legend(title="Labels")
plt.title("umap with Euclidean Distance")
plt.savefig(f"umap_euclidean.pdf", bbox_inches='tight')
plt.show()

## SWKL

In [ ]:
X = df_no_dup.traj_counts_perturbed_normalised.values
X_counts = df_no_dup.traj_counts_perturbed.values
labels = df_no_dup.russian.values

In [ ]:
def compute_row_swkl(i, X, X_counts):
    n = len(X)
    row = np.zeros(n)
    for j in range(i+1, n):
        row[j] = weighted_symmetric_kl_divergence(X[i], X[j], X_counts[i], X_counts[j])
    return i, row

# X is array of matrices, dtype=object
n = len(X)
dist_matrix = np.zeros((n, n))


# for i in range(n): 
#     for j in range(i+1, n): 
#         d = weighted_symmetric_kl_divergence(X[i], X[j], X_counts[i], X_counts[j])
#         dist_matrix[i, j] = d 
#         dist_matrix[j, i] = d

# Parallelize rows
results = Parallel(n_jobs=-1, backend="loky")(
    delayed(compute_row_swkl)(i, X, X_counts) for i in range(n)
)


# Fill results back into matrix
for i, row in results:
    dist_matrix[i, i+1:] = row[i+1:]
    dist_matrix[i+1:, i] = row[i+1:]

In [ ]:

# Run t-SNE with precomputed distances
tsne = TSNE(n_components=2, metric="precomputed", init="random", random_state=42)
X_embedded = tsne.fit_transform(dist_matrix)


In [ ]:
colors = np.array(['blue', 'red'])  # map 0 -> blue, 1 -> red
point_colors = colors[labels]

plt.figure(figsize=(8,6))
plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=point_colors,s=40, alpha=0.2)
# Create a legend
for label, color in zip(['organic', 'troll'],['blue', 'red']):
    plt.scatter([], [], c=color, label=f'{label}')  # empty scatter for legend
plt.legend(title="Labels")
plt.title("t-SNE with SWKL")
plt.savefig(f"tsne_swkl.pdf", bbox_inches='tight')
plt.show()


# Run UMAP with precomputed distances
umap_model = umap.UMAP(
    n_components=2,
    metric="precomputed",
    random_state=42
)
X_embedded = umap_model.fit_transform(dist_matrix)
colors = np.array(['blue', 'red'])  # map 0 -> blue, 1 -> red
point_colors = colors[labels]

plt.figure(figsize=(8,6))
plt.scatter(X_embedded[:, 0], X_embedded[:, 1], c=point_colors,s=40, alpha=0.2)
# Create a legend
for label, color in zip(['organic', 'troll'],['blue', 'red']):
    plt.scatter([], [], c=color, label=f'{label}')  # empty scatter for legend
plt.legend(title="Labels")
plt.title("umap with SWKL")
plt.savefig(f"umap_swkl.pdf", bbox_inches='tight')
plt.show()